# PFE ML - Model Interrogation By SIREN

This Colab workbook loads the continuity-risk ML artifacts from Google Drive, ranks the trained runs, selects the best deployable model artifact, then scores one or more French company SIRENs.

Default storage root used by the training notebooks:

`/content/drive/MyDrive/PFE ML Data/pfe_data`

The prediction target is `continuity_risk_12m_label`: whether a company is likely to stop being active/open within the next 12 months from the selected feature cutoff year.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = Path('/content/pfein')
BACKEND_DIR = REPO_DIR / 'back_end'

DRIVE_ROOT = Path('/content/drive/MyDrive/PFE ML Data/pfe_data')
DUCKDB_TMP = Path('/content/pfein_duckdb_tmp')
INSTALL_REQUIREMENTS = True

cwd = Path.cwd()
if (cwd / 'collabs' / 'requirements-colab.txt').exists():
    BACKEND_DIR = cwd
    REPO_DIR = BACKEND_DIR.parent

if not (BACKEND_DIR / 'collabs' / 'requirements-colab.txt').exists():
    if not (REPO_DIR / '.git').exists():
        subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'switch', BRANCH])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])

if not (BACKEND_DIR / 'collabs' / 'requirements-colab.txt').exists():
    raise FileNotFoundError(f'Backend repository is incomplete: {BACKEND_DIR}')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DUCKDB_TMP.mkdir(parents=True, exist_ok=True)
os.environ['DUCKDB_TEMP_DIRECTORY'] = str(DUCKDB_TMP)

os.chdir(BACKEND_DIR)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

if INSTALL_REQUIREMENTS:
    requirements = BACKEND_DIR / 'collabs' / 'requirements-colab.txt'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])

print(f'BACKEND_DIR = {BACKEND_DIR}')
print(f'DRIVE_ROOT  = {DRIVE_ROOT}')
print(f'BRANCH      = {BRANCH}')

## 1. Inspect Artifacts And Select The Deployable Model

The comparison table ranks all recorded runs. To keep the comparison fair, the default ranking is restricted to the same temporal split as the deployed metadata. If a run-specific `*.joblib` exists under `ml-artifacts/runs/<run_name>/`, the notebook can load that archived model. Otherwise it uses the root `ml-artifacts/model.joblib`, which is the deployable artifact produced by the latest training run.

In [ ]:
import json
import pandas as pd
from IPython.display import display

ARTIFACTS_DIR = DRIVE_ROOT / 'ml-artifacts'
DATA_LAKE = DRIVE_ROOT / 'data-lake'
MODEL_PATH = ARTIFACTS_DIR / 'model.joblib'
METADATA_PATH = ARTIFACTS_DIR / 'model_metadata.json'
COMPARISON_PATH = ARTIFACTS_DIR / 'model_run_comparison.csv'

SELECTION_METRIC = 'average_precision'
RESTRICT_TO_DEPLOYMENT_SPLIT = True

for required_path in [MODEL_PATH, METADATA_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required artifact: {required_path}')

metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
comparison = pd.read_csv(COMPARISON_PATH) if COMPARISON_PATH.exists() else pd.DataFrame()


def _select_model_in(run_dir):
    if not run_dir or not run_dir.exists():
        return None, None
    for candidate in (run_dir / 'model.joblib', run_dir / 'model.pkl'):
        if candidate.exists():
            return candidate, run_dir / 'metadata.json'
    matches = sorted(run_dir.glob('*.joblib')) + sorted(run_dir.glob('*.pkl'))
    if matches:
        return matches[0], run_dir / 'metadata.json'
    return None, None


def resolve_archived_model(run_name, run_artifacts_dir=None):
    # Supports both layouts:
    #   ml-artifacts/runs/<run_name>/            (legacy)
    #   ml-artifacts/runs/<family>/<run_name>/   (current, family-grouped)
    # The full run_artifacts_dir from the run index is preferred when available.
    if isinstance(run_artifacts_dir, str) and run_artifacts_dir:
        chosen, meta = _select_model_in(Path(run_artifacts_dir))
        if chosen is not None:
            return chosen, meta
    if not isinstance(run_name, str) or not run_name:
        return None, None
    chosen, meta = _select_model_in(ARTIFACTS_DIR / 'runs' / run_name)
    if chosen is not None:
        return chosen, meta
    runs_root = ARTIFACTS_DIR / 'runs'
    if runs_root.exists():
        for family_dir in runs_root.iterdir():
            if family_dir.is_dir():
                chosen, meta = _select_model_in(family_dir / run_name)
                if chosen is not None:
                    return chosen, meta
    return None, None


SELECTED_MODEL_PATH = MODEL_PATH
SELECTED_METADATA_PATH = METADATA_PATH
selected_reason = 'Using root ml-artifacts/model.joblib because it is the deployable artifact produced by the latest training run.'

if not comparison.empty:
    metric = SELECTION_METRIC if SELECTION_METRIC in comparison.columns else 'roc_auc'
    pool = comparison.copy()
    if RESTRICT_TO_DEPLOYMENT_SPLIT and 'split_strategy' in pool.columns:
        pool = pool[pool['split_strategy'].eq(metadata.get('split_strategy'))].copy()
    if pool.empty:
        pool = comparison.copy()
    sort_columns = [c for c in [metric, 'roc_auc', 'trained_at'] if c in pool.columns]
    ascending = [False for _ in sort_columns]
    ranking = pool.sort_values(sort_columns, ascending=ascending) if sort_columns else pool
    best_row = ranking.iloc[0]
    for _, row in ranking.iterrows():
        candidate_model, candidate_metadata = resolve_archived_model(
            row.get('run_name'),
            run_artifacts_dir=row.get('run_artifacts_dir'),
        )
        if candidate_model is not None:
            SELECTED_MODEL_PATH = candidate_model
            SELECTED_METADATA_PATH = candidate_metadata if candidate_metadata and candidate_metadata.exists() else METADATA_PATH
            selected_reason = f'Using archived model for best ranked deployable run: {row.get("run_name")}'
            break
    display_columns = [
        'model_version', 'run_name', 'model_family', 'split_strategy',
        'average_precision', 'roc_auc', 'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5'
    ]
    display_columns = [c for c in display_columns if c in ranking.columns]
    print('Top runs by selected ranking policy:')
    display(ranking[display_columns].head(10))
    print('Best run in comparison table:', best_row.get('run_name'))
    print(f'Best {metric}:', best_row.get(metric))

selected_metadata = json.loads(SELECTED_METADATA_PATH.read_text(encoding='utf-8'))
print('
Selected model artifact:', SELECTED_MODEL_PATH)
print('Selected metadata:', SELECTED_METADATA_PATH)
print('Selection reason:', selected_reason)
print('Model version:', selected_metadata.get('model_version'))
print('Run name:', selected_metadata.get('run_name'))
print('Family:', selected_metadata.get('model_family'))
print('Average precision:', selected_metadata.get('metrics', {}).get('average_precision'))
print('ROC AUC:', selected_metadata.get('metrics', {}).get('roc_auc'))


## 2. Load Model And Define SIREN Scoring Helpers

In [ ]:
import re
import sys
import duckdb
import joblib
import numpy as np
import pandas as pd

# The training script was executed as __main__ when the model was fit, so the
# pickled pipeline references __main__.CategoricalCardinalityCapper (and other
# custom classes). Importing the module is not enough — pickle looks for those
# names on __main__ specifically. Re-alias every class defined in the training
# module onto __main__ before joblib.load() so unpickling can resolve them.
from app.tools import train_continuity_model as _train_continuity_model  # noqa: F401
_main_module = sys.modules['__main__']
for _name in dir(_train_continuity_model):
    _obj = getattr(_train_continuity_model, _name)
    if isinstance(_obj, type) and getattr(_obj, '__module__', None) == _train_continuity_model.__name__:
        setattr(_main_module, _name, _obj)

# train_continuity_model.py saves a bundle dict, not the raw estimator:
#   {'pipeline': <sklearn pipeline>, 'feature_columns': [...], 'target': ..., ...}
# Unwrap it, and fall back to the loaded object directly if an older artifact
# happens to be a bare estimator.
_loaded_artifact = joblib.load(SELECTED_MODEL_PATH)
if isinstance(_loaded_artifact, dict) and 'pipeline' in _loaded_artifact:
    model = _loaded_artifact['pipeline']
    bundle_feature_columns = _loaded_artifact.get('feature_columns')
else:
    model = _loaded_artifact
    bundle_feature_columns = None

calibrator = None
calibrator_path = None
calibrator_candidates = [
    SELECTED_MODEL_PATH.parent / 'isotonic.joblib',
    SELECTED_MODEL_PATH.parent / 'isotonic_calibrator.joblib',
    SELECTED_MODEL_PATH.parent / 'platt.joblib',
    ARTIFACTS_DIR / 'isotonic.joblib',
    ARTIFACTS_DIR / 'isotonic_calibrator.joblib',
    ARTIFACTS_DIR / 'platt.joblib',
]
for candidate in calibrator_candidates:
    if candidate.exists():
        calibrator_path = candidate
        calibrator = joblib.load(candidate)
        break

if calibrator_path:
    print(f'Loaded probability calibrator: {calibrator_path}')
else:
    print('No probability calibrator found. Raw scores are useful for ranking; absolute probabilities may be overestimated.')

FEATURES_PATH = DATA_LAKE / 'features' / 'company_year_features'
FEATURES_GLOB = (FEATURES_PATH / '**' / '*.parquet').as_posix()
_FEATURES_GLOB_SQL = FEATURES_GLOB.replace(chr(39), chr(39) + chr(39))

if not FEATURES_PATH.exists():
    raise FileNotFoundError(f'Missing feature dataset: {FEATURES_PATH}')

def sql_literal(value):
    value = str(value).replace(chr(39), chr(39) + chr(39))
    return chr(39) + value + chr(39)

def normalize_siren(value):
    digits = re.sub(r'\D', '', str(value))
    if len(digits) != 9:
        raise ValueError(f'SIREN must contain exactly 9 digits, got {value!r}')
    return digits

def available_years_for_siren(siren):
    siren = normalize_siren(siren)
    query = f'''
        SELECT DISTINCT prediction_year
        FROM read_parquet('{_FEATURES_GLOB_SQL}', union_by_name=true)
        WHERE CAST(siren AS VARCHAR) = {sql_literal(siren)}
        ORDER BY prediction_year
    '''
    con = duckdb.connect()
    try:
        years = con.execute(query).df()['prediction_year'].dropna().astype(int).tolist()
    finally:
        con.close()
    return years

def load_company_feature_row(siren, prediction_year=None):
    siren = normalize_siren(siren)
    year_clause = '' if prediction_year is None else f'AND prediction_year = {int(prediction_year)}'
    query = f'''
        SELECT *
        FROM read_parquet('{_FEATURES_GLOB_SQL}', union_by_name=true)
        WHERE CAST(siren AS VARCHAR) = {sql_literal(siren)}
        {year_clause}
        ORDER BY prediction_year DESC
        LIMIT 1
    '''
    con = duckdb.connect()
    try:
        df = con.execute(query).df()
    finally:
        con.close()
    if df.empty:
        years = available_years_for_siren(siren)
        if years:
            raise ValueError(f'No feature row for SIREN {siren} and year {prediction_year}. Available years: {years}')
        raise ValueError(f'No feature row found for SIREN {siren}. Rebuild company_year_features if needed.')
    return df

def expected_feature_columns():
    # The bundle is the most authoritative source — it was written alongside the
    # fitted pipeline. Fall back to the separate metadata file, then to the
    # estimator's own attributes.
    if bundle_feature_columns:
        return list(bundle_feature_columns)
    columns = selected_metadata.get('feature_columns')
    if columns:
        return list(columns)
    if hasattr(model, 'feature_names_in_'):
        return list(model.feature_names_in_)
    if hasattr(model, 'named_steps'):
        for step in model.named_steps.values():
            if hasattr(step, 'feature_names_in_'):
                return list(step.feature_names_in_)
    raise RuntimeError('Could not determine model feature columns from metadata or fitted pipeline.')

FEATURE_COLUMNS = expected_feature_columns()
print(f'Model expects {len(FEATURE_COLUMNS)} feature columns.')

def align_features_for_model(df):
    aligned = df.copy()
    missing = [column for column in FEATURE_COLUMNS if column not in aligned.columns]
    for column in missing:
        aligned[column] = np.nan
    for column in aligned.columns:
        if pd.api.types.is_bool_dtype(aligned[column]):
            aligned[column] = aligned[column].astype(float)
    if missing:
        print('Missing model columns were added as NaN:', missing)
    return aligned[FEATURE_COLUMNS]

def maybe_calibrate(raw_probability):
    if calibrator is None:
        return None
    raw_probability = np.asarray(raw_probability, dtype=float)
    try:
        return calibrator.predict(raw_probability)
    except Exception:
        return calibrator.predict(raw_probability.reshape(-1, 1))

def risk_band(probability):
    if pd.isna(probability):
        return 'unknown'
    probability = float(probability)
    if probability >= 0.50:
        return 'high'
    if probability >= 0.30:
        return 'medium'
    if probability >= 0.10:
        return 'watch'
    return 'low'

def score_siren(siren, prediction_year=None):
    df = load_company_feature_row(siren, prediction_year)
    X = align_features_for_model(df)
    raw = model.predict_proba(X)[:, 1].astype(float)
    calibrated = maybe_calibrate(raw)
    out = pd.DataFrame(index=df.index)
    passthrough_columns = [
        'siren', 'prediction_year', 'prediction_date', 'company_name', 'activity_code',
        'legal_category_code', 'employee_size_bracket', 'administrative_status_at_cutoff',
        'company_age_years', 'legal_events_count_all', 'legal_risk_events_count_all',
        'legal_events_count_12m', 'radiation_events_count_all', 'days_since_last_legal_event',
        'annual_accounts_count_all', 'has_financial_data', 'latest_revenue', 'latest_net_result',
        'latest_debt_to_assets', 'years_since_last_financial_statement'
    ]
    for column in passthrough_columns:
        if column in df.columns:
            out[column] = df[column].values
    out['continuity_risk_12m_score_raw'] = raw
    if calibrated is not None:
        out['continuity_risk_12m_score_calibrated'] = np.asarray(calibrated, dtype=float)
        score_for_decision = out['continuity_risk_12m_score_calibrated']
    else:
        score_for_decision = out['continuity_risk_12m_score_raw']
    out['decision_threshold_0_5'] = score_for_decision >= 0.5
    out['risk_band'] = score_for_decision.map(risk_band)
    out['selected_model_version'] = selected_metadata.get('model_version')
    out['selected_run_name'] = selected_metadata.get('run_name')
    train_end = selected_metadata.get('train_end_year')
    scored_year = int(out['prediction_year'].iloc[0]) if 'prediction_year' in out.columns else None
    if train_end is not None and scored_year is not None and scored_year > int(train_end):
        print(f'Warning: scoring prediction_year={scored_year}, which is after the training end year {train_end}.')
    return out.reset_index(drop=True)

def display_prediction_table(result):
    columns = [
        'siren', 'prediction_year', 'company_name', 'activity_code', 'legal_category_code',
        'administrative_status_at_cutoff', 'company_age_years',
        'continuity_risk_12m_score_raw', 'continuity_risk_12m_score_calibrated',
        'decision_threshold_0_5', 'risk_band'
    ]
    columns = [column for column in columns if column in result.columns]
    probability_columns = [column for column in columns if column.startswith('continuity_risk')]
    formatters = {column: '{:.2%}' for column in probability_columns}
    display(result[columns].style.format(formatters))

def predict_siren(siren, prediction_year=None):
    result = score_siren(siren, prediction_year)
    display_prediction_table(result)
    return result

def predict_many_sirens(sirens, prediction_year=None):
    frames = []
    errors = []
    for value in sirens:
        try:
            frames.append(score_siren(value, prediction_year))
        except Exception as exc:
            errors.append({'siren': str(value), 'error': str(exc)})
    result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not result.empty:
        sort_column = (
            'continuity_risk_12m_score_calibrated'
            if 'continuity_risk_12m_score_calibrated' in result.columns
            else 'continuity_risk_12m_score_raw'
        )
        result = result.sort_values(sort_column, ascending=False).reset_index(drop=True)
        display_prediction_table(result)
    if errors:
        print('SIRENs not scored:')
        display(pd.DataFrame(errors))
    return result


## 2b. Conformal confidence (the `confiance` field)

Loads the Mondrian conformal predictor bundled with the model (schema ≥ 2.1) and adds two per-company numbers: **confidence** (how sure we are of the predicted class) and **credibility** (how well this company resembles the training data). Call `predict_siren_with_confidence(siren)` instead of `predict_siren`. This is exactly the value the backend now publishes as `confidence`/`credibility` on the prediction document.

In [ ]:
# --- Conformal confidence (per-company) -------------------------------------
# Loads the Mondrian conformal predictor saved with the model (schema >= 2.1).
# Adds `confidence` (how sure we are of the predicted class) and `credibility`
# (how well the company fits the training distribution). This is the value
# behind the product's `confiance` field. The class unpickles because the cell
# above re-aliased every train_continuity_model class onto __main__.
conformal = None
if isinstance(_loaded_artifact, dict):
    conformal = _loaded_artifact.get('conformal')
if conformal is None:
    for _cand in [SELECTED_MODEL_PATH.parent / 'conformal_calibrator.joblib',
                  ARTIFACTS_DIR / 'conformal_calibrator.joblib']:
        if _cand.exists():
            conformal = joblib.load(_cand)
            break
print('Conformal predictor:', 'loaded' if conformal is not None else 'not available (train a schema >= 2.1 model first)')


def add_confidence(result):
    if conformal is None or result.empty:
        return result
    cp = conformal.predict(result['continuity_risk_12m_score_raw'].to_numpy())
    out = result.copy()
    out['confidence'] = cp['confidence']
    out['credibility'] = cp['credibility']
    return out


def predict_siren_with_confidence(siren, prediction_year=None):
    result = add_confidence(score_siren(siren, prediction_year))
    columns = [
        'siren', 'prediction_year', 'company_name',
        'continuity_risk_12m_score_raw', 'continuity_risk_12m_score_calibrated',
        'confidence', 'credibility', 'risk_band',
    ]
    columns = [c for c in columns if c in result.columns]
    pct_columns = [c for c in columns if 'score' in c or c in ('confidence', 'credibility')]
    display(result[columns].style.format({c: '{:.2%}' for c in pct_columns}))
    return result

# Example:
# predict_siren_with_confidence('552120222')

## 3. Score One Company

Set `SIREN` to the 9-digit company identifier. Leave `PREDICTION_YEAR = None` to use the newest feature row available for that SIREN, or set a year such as `2024` for a fixed cutoff.

In [ ]:
SIREN = ''  # Example: '552120222'
PREDICTION_YEAR = None

if SIREN:
    prediction = predict_siren(SIREN, PREDICTION_YEAR)
else:
    print('Set SIREN to a 9-digit value, then run this cell.')

## 4. Optional Batch Scoring

In [ ]:
SIRENS = []  # Example: ['552120222', '542065305']
BATCH_PREDICTION_YEAR = None

if SIRENS:
    batch_predictions = predict_many_sirens(SIRENS, BATCH_PREDICTION_YEAR)
else:
    print('Add one or more SIRENs to SIRENS, then run this cell.')

## Interpretation Note

If no calibrator is present, the raw score should be treated mainly as a ranking signal. The training metadata shows that the model uses class balancing for a rare target, so raw probabilities can be higher than observed event rates. For user-facing probability text, add and load an isotonic or Platt calibrator from the calibration phase.

## 5. Rules Layer (Post-Cutoff Events)

The model only sees features as of its `prediction_year` cutoff (Dec 31 of that year). Anything that happened after — e.g. a BODACC radiation in 2025 or an INPI cessation in 2026 — is invisible to the ML score.

The cell below queries the data lake for events that occurred between the prediction cutoff and today, then applies a rule set that mirrors the labels the model was trained on:

| Rule | Trigger | Effect |
|---|---|---|
| R1 | Current `administrative_status` is closed | bucket → `already_closed`, probability = 100% |
| R2 | INPI radiation/cessation formality after cutoff | bucket → at least `high`, probability ≥ 85% |
| R3 | BODACC radiation after cutoff | bucket → at least `high`, probability ≥ 85% |
| R4 | BODACC procédure collective after cutoff | bucket → at least `high`, probability ≥ 70% |
| R5 | ≥ 2 legal risk events in last 90 days | bump bucket up one level |
| R6 | No annual filing in last 18 months, had prior filings | bump bucket up one level |

The ML score is preserved unchanged (auditable); `adjusted_*` columns show the post-rules result. Use `predict_siren_with_rules` / `predict_many_sirens_with_rules` instead of the plain versions.

In [ ]:
from datetime import date
from pathlib import Path

def _resolve_dataset_glob(*candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if not candidate.exists():
            continue
        if any(candidate.rglob('*.parquet')):
            glob_path = (candidate / '**' / '*.parquet').as_posix()
            return glob_path.replace("'", "''"), candidate
    return None, None

_CLEAN = DATA_LAKE / 'clean'
_RAW = DATA_LAKE / 'raw'

LEGAL_EVENTS_SQL, LEGAL_EVENTS_DIR = _resolve_dataset_glob(
    _CLEAN / 'legal_events', _RAW / 'bodacc'
)
FORMALITIES_SQL, FORMALITIES_DIR = _resolve_dataset_glob(
    _CLEAN / 'formalities_events', _RAW / 'inpi' / 'formalites'
)
COMPANY_IDENTITY_SQL, COMPANY_IDENTITY_DIR = _resolve_dataset_glob(
    _CLEAN / 'company_identity', _RAW / 'insee' / 'unites_legales'
)
ANNUAL_ACCOUNTS_SQL, ANNUAL_ACCOUNTS_DIR = _resolve_dataset_glob(
    _CLEAN / 'annual_accounts', _RAW / 'inpi' / 'comptes_annuels'
)

print('Rules layer datasets resolved:')
for name, path in [
    ('legal_events (BODACC)', LEGAL_EVENTS_DIR),
    ('formalities_events (INPI)', FORMALITIES_DIR),
    ('company_identity (INSEE)', COMPANY_IDENTITY_DIR),
    ('annual_accounts (INPI)', ANNUAL_ACCOUNTS_DIR),
]:
    print(f'  {name}: {path or "NOT FOUND — rules using this dataset will skip"}')

RISK_LEVELS = ['low', 'watch', 'medium', 'high']
CLOSED_STATUSES = {'C', 'CESSEE', 'FERMEE', 'INACTIVE'}

def _bump_bucket(bucket, levels=1):
    if bucket == 'already_closed' or bucket not in RISK_LEVELS:
        return bucket
    idx = min(len(RISK_LEVELS) - 1, RISK_LEVELS.index(bucket) + levels)
    return RISK_LEVELS[idx]

def _max_bucket(a, b):
    if 'already_closed' in (a, b):
        return 'already_closed'
    if a not in RISK_LEVELS:
        return b if b in RISK_LEVELS else a
    if b not in RISK_LEVELS:
        return a
    return RISK_LEVELS[max(RISK_LEVELS.index(a), RISK_LEVELS.index(b))]

def _safe_query(con, sql):
    try:
        return con.execute(sql).df()
    except Exception as exc:
        print(f'  Skipping rule query (missing columns or dataset issue): {exc}')
        return pd.DataFrame()

def query_post_cutoff_evidence(siren, cutoff_date):
    siren_lit = sql_literal(normalize_siren(siren))
    cutoff_lit = sql_literal(str(cutoff_date))
    today_lit = sql_literal(str(date.today()))
    con = duckdb.connect()
    try:
        current_status = None
        if COMPANY_IDENTITY_SQL:
            df = _safe_query(con, f'''
                SELECT administrative_status
                FROM read_parquet('{COMPANY_IDENTITY_SQL}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY COALESCE(period_start, DATE '1900-01-01') DESC
                LIMIT 1
            ''')
            if not df.empty:
                current_status = df.iloc[0, 0]
        bodacc_events = pd.DataFrame()
        if LEGAL_EVENTS_SQL:
            bodacc_events = _safe_query(con, f'''
                SELECT event_date,
                       event_category,
                       event_type,
                       COALESCE(is_risk_event, FALSE) AS is_risk_event,
                       COALESCE(is_radiation, FALSE) AS is_radiation,
                       COALESCE(flag_liquidation, FALSE) AS flag_liquidation,
                       COALESCE(flag_redressement, FALSE) AS flag_redressement,
                       COALESCE(flag_sauvegarde, FALSE) AS flag_sauvegarde,
                       COALESCE(flag_procedure_collective, FALSE) AS flag_procedure_collective,
                       COALESCE(flag_cessation_paiement, FALSE) AS flag_cessation_paiement
                FROM read_parquet('{LEGAL_EVENTS_SQL}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                  AND event_date > DATE {cutoff_lit}
                  AND event_date <= DATE {today_lit}
                ORDER BY event_date DESC
            ''')
        inpi_radiations = pd.DataFrame()
        if FORMALITIES_SQL:
            inpi_radiations = _safe_query(con, f'''
                SELECT event_date, event_type, event_text
                FROM read_parquet('{FORMALITIES_SQL}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                  AND event_date > DATE {cutoff_lit}
                  AND event_date <= DATE {today_lit}
                  AND (
                    lower(COALESCE(event_type, '') || ' ' || COALESCE(event_text, '')) LIKE '%cessation%'
                    OR lower(COALESCE(event_type, '') || ' ' || COALESCE(event_text, '')) LIKE '%radiation%'
                    OR lower(COALESCE(event_type, '') || ' ' || COALESCE(event_text, '')) LIKE '%fermeture%'
                  )
                ORDER BY event_date DESC
            ''')
        last_filing_date = None
        any_prior_filing = False
        if ANNUAL_ACCOUNTS_SQL:
            df = _safe_query(con, f'''
                SELECT max(filing_date) AS last_filing,
                       count(*) FILTER (WHERE filing_date <= DATE {cutoff_lit}) AS prior_count
                FROM read_parquet('{ANNUAL_ACCOUNTS_SQL}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                  AND filing_date <= DATE {today_lit}
            ''')
            if not df.empty:
                last_filing_date = df.iloc[0]['last_filing']
                any_prior_filing = int(df.iloc[0]['prior_count'] or 0) > 0
    finally:
        con.close()
    return {
        'current_status': current_status,
        'bodacc_events': bodacc_events,
        'inpi_radiations': inpi_radiations,
        'last_filing_date': last_filing_date,
        'any_prior_filing': any_prior_filing,
    }

def evaluate_rules(siren, cutoff_date, ml_probability, ml_bucket):
    evidence = query_post_cutoff_evidence(siren, cutoff_date)
    firings = []
    today = pd.Timestamp(date.today())
    status_raw = evidence['current_status']
    status = (str(status_raw) if status_raw is not None else '').strip().upper()
    if status in CLOSED_STATUSES:
        firings.append({
            'code': 'R1_ALREADY_CLOSED',
            'label': 'Already administratively closed',
            'severity': 'critical',
            'evidence_date': None,
            'description': f'Current administrative status is {status_raw!r}',
        })
        return {
            'firings': firings,
            'final_probability': 1.0,
            'final_bucket': 'already_closed',
            'is_overridden': True,
            'override_reason': 'Company is administratively closed',
        }
    final_prob = float(ml_probability) if not pd.isna(ml_probability) else 0.0
    final_bucket = ml_bucket
    if not evidence['inpi_radiations'].empty:
        row = evidence['inpi_radiations'].iloc[0]
        firings.append({
            'code': 'R2_INPI_RADIATION',
            'label': 'INPI radiation/cessation formality after cutoff',
            'severity': 'high',
            'evidence_date': row['event_date'],
            'description': f"INPI {row['event_type']!r} on {row['event_date']}",
        })
        final_prob = max(final_prob, 0.85)
        final_bucket = _max_bucket(final_bucket, 'high')
    bodacc = evidence['bodacc_events']
    if not bodacc.empty:
        radiation = bodacc[bodacc['is_radiation'] == True]
        if not radiation.empty:
            row = radiation.iloc[0]
            firings.append({
                'code': 'R3_BODACC_RADIATION',
                'label': 'BODACC radiation after cutoff',
                'severity': 'high',
                'evidence_date': row['event_date'],
                'description': f"BODACC radiation {row['event_type']!r} on {row['event_date']}",
            })
            final_prob = max(final_prob, 0.85)
            final_bucket = _max_bucket(final_bucket, 'high')
        procedure = bodacc[
            (bodacc['flag_liquidation'] == True)
            | (bodacc['flag_redressement'] == True)
            | (bodacc['flag_sauvegarde'] == True)
            | (bodacc['flag_procedure_collective'] == True)
            | (bodacc['flag_cessation_paiement'] == True)
        ]
        if not procedure.empty:
            row = procedure.iloc[0]
            procedure_names = [
                name for flag, name in [
                    ('flag_liquidation', 'liquidation'),
                    ('flag_redressement', 'redressement'),
                    ('flag_sauvegarde', 'sauvegarde'),
                    ('flag_procedure_collective', 'procédure collective'),
                    ('flag_cessation_paiement', 'cessation paiement'),
                ] if row.get(flag)
            ]
            firings.append({
                'code': 'R4_BODACC_PROCEDURE_COLLECTIVE',
                'label': 'BODACC procédure collective after cutoff',
                'severity': 'high',
                'evidence_date': row['event_date'],
                'description': f"{', '.join(procedure_names) or 'procédure'} on {row['event_date']}",
            })
            final_prob = max(final_prob, 0.70)
            final_bucket = _max_bucket(final_bucket, 'high')
        recent_window_start = today - pd.Timedelta(days=90)
        risk_events = bodacc[
            (bodacc['is_risk_event'] == True)
            & (pd.to_datetime(bodacc['event_date']) >= recent_window_start)
        ]
        if len(risk_events) >= 2:
            firings.append({
                'code': 'R5_MULTIPLE_RISK_EVENTS',
                'label': f'{len(risk_events)} legal risk events in last 90 days',
                'severity': 'medium',
                'evidence_date': risk_events.iloc[0]['event_date'],
                'description': f"{len(risk_events)} BODACC risk events between "
                               f"{risk_events['event_date'].min()} and {risk_events['event_date'].max()}",
            })
            final_bucket = _bump_bucket(final_bucket, 1)
    if evidence['any_prior_filing']:
        last_filing = pd.to_datetime(evidence['last_filing_date']) if evidence['last_filing_date'] is not None else None
        eighteen_months_ago = today - pd.Timedelta(days=18 * 30)
        if last_filing is None or last_filing < eighteen_months_ago:
            firings.append({
                'code': 'R6_MISSED_FILING',
                'label': 'No annual filing in last 18 months',
                'severity': 'medium',
                'evidence_date': last_filing.date() if last_filing is not None else None,
                'description': f"Last filing: {last_filing.date() if last_filing is not None else 'none'}; expected within 18 months",
            })
            final_bucket = _bump_bucket(final_bucket, 1)
    is_overridden = (final_bucket != ml_bucket) or (final_prob > float(ml_probability or 0))
    override_reason = (
        f"Rules adjusted bucket {ml_bucket} -> {final_bucket}: " + ', '.join(f['code'] for f in firings)
        if is_overridden and firings else None
    )
    return {
        'firings': firings,
        'final_probability': final_prob,
        'final_bucket': final_bucket,
        'is_overridden': is_overridden,
        'override_reason': override_reason,
    }

def score_siren_with_rules(siren, prediction_year=None):
    base = score_siren(siren, prediction_year)
    row = base.iloc[0]
    cutoff_date = date(int(row['prediction_year']), 12, 31)
    ml_prob_col = (
        'continuity_risk_12m_score_calibrated'
        if 'continuity_risk_12m_score_calibrated' in base.columns
        else 'continuity_risk_12m_score_raw'
    )
    ml_prob = float(row[ml_prob_col])
    ml_bucket = row['risk_band']
    rules_result = evaluate_rules(siren, cutoff_date, ml_prob, ml_bucket)
    enriched = base.copy()
    enriched['adjusted_probability'] = rules_result['final_probability']
    enriched['adjusted_risk_bucket'] = rules_result['final_bucket']
    enriched['is_overridden'] = rules_result['is_overridden']
    enriched['override_reason'] = rules_result['override_reason']
    return enriched, rules_result['firings']

def display_prediction_with_rules(enriched, firings):
    columns = [
        'siren', 'prediction_year', 'company_name',
        'administrative_status_at_cutoff',
        'continuity_risk_12m_score_calibrated', 'risk_band',
        'adjusted_probability', 'adjusted_risk_bucket',
        'is_overridden', 'override_reason',
    ]
    columns = [c for c in columns if c in enriched.columns]
    prob_columns = [c for c in columns if 'probability' in c or 'score' in c]
    formatters = {c: '{:.2%}' for c in prob_columns}
    display(enriched[columns].style.format(formatters))
    if firings:
        print('Rules fired:')
        display(pd.DataFrame(firings))
    else:
        print('No rules fired — ML score is the final score.')

def predict_siren_with_rules(siren, prediction_year=None):
    enriched, firings = score_siren_with_rules(siren, prediction_year)
    display_prediction_with_rules(enriched, firings)
    return enriched

def predict_many_sirens_with_rules(sirens, prediction_year=None):
    frames = []
    all_firings = []
    errors = []
    for value in sirens:
        try:
            enriched, firings = score_siren_with_rules(value, prediction_year)
            frames.append(enriched)
            for firing in firings:
                all_firings.append({'siren': str(value), **firing})
        except Exception as exc:
            errors.append({'siren': str(value), 'error': str(exc)})
    result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not result.empty:
        sort_col = (
            'adjusted_probability' if 'adjusted_probability' in result.columns
            else 'continuity_risk_12m_score_calibrated' if 'continuity_risk_12m_score_calibrated' in result.columns
            else 'continuity_risk_12m_score_raw'
        )
        result = result.sort_values(sort_col, ascending=False).reset_index(drop=True)
        display_columns = [
            'siren', 'prediction_year', 'company_name',
            'continuity_risk_12m_score_calibrated', 'risk_band',
            'adjusted_probability', 'adjusted_risk_bucket',
            'is_overridden', 'override_reason',
        ]
        display_columns = [c for c in display_columns if c in result.columns]
        prob_columns = [c for c in display_columns if 'probability' in c or 'score' in c]
        formatters = {c: '{:.2%}' for c in prob_columns}
        display(result[display_columns].style.format(formatters))
    if all_firings:
        print('All rule firings:')
        display(pd.DataFrame(all_firings))
    if errors:
        print('SIRENs not scored:')
        display(pd.DataFrame(errors))
    return result

# Example usage — replace SIREN and run:
# predict_siren_with_rules('901399675')
# predict_many_sirens_with_rules(['901399675', '552120222'])
